In [1]:
"""
Script to calculate C4MIP beta & gamma values for CMIP6 data.

Land:

"""

import numpy as np
import iris
from iris import cube
import iris.coord_categorisation
import iris.analysis.cartography
import glob
import warnings
from iris.util import equalise_attributes 
from iris.util import unify_time_units
import pandas as pd

In [2]:
# CMIP6 data processing

def cmip6_tcre_processing(directory, model):

    """
    Takes CMIP6 netcdf files and produces single cube containing annual averages.

    """

    # Combine files into single cube
    list_files = glob.glob(directory)
    list_files = np.array(list_files)
    newlist = np.sort(list_files)

    Cubelist = iris.cube.CubeList([])

    for i in range(0, len(newlist)):

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            warnings.simplefilter('ignore', UserWarning)
            # warnings.simplefilter('ignore', IrisDefaultingWarning)
            
            cube = iris.load_cube(newlist[i])

            cube.coord('latitude').attributes = {}
            cube.coord('longitude').attributes = {}  
            if i == 0:
                metadata1 = cube.metadata
            else:
                cube.metadata = metadata1
            
            if model=='IPSL-CM6A-LR' or model=='CNRM-ESM2-1':
                 cube.coord('latitude').guess_bounds()
                 cube.coord('longitude').guess_bounds()
         
            # CESM2 bound issue fix
            if (model=='CESM2') & (i==0):
                lat_data = cube.coord('latitude').points
                lon_data = cube.coord('longitude').points
                lat_bounds = cube.coord('latitude').bounds
                lon_bounds = cube.coord('longitude').bounds
            elif (model=='CESM2') & (i>0):
                cube.coord('latitude').points = lat_data
                cube.coord('longitude').points = lon_data
                cube.coord('latitude').bounds = lat_bounds
                cube.coord('longitude').bounds = lon_bounds
    
            if model=='IPSL-CM6A-LR':
                cube.coord('time').attributes.pop('time_origin')
            
            Cubelist.append(cube)

    unify_time_units(Cubelist)
    equalise_attributes(Cubelist)

    for cube in Cubelist:
        lon_bounds = Cubelist[0].coord('longitude').bounds
        cube.coord('longitude').bounds = lon_bounds

    for i, cube in enumerate(Cubelist):
        if cube.coord('time').units == Cubelist[0].coord('time').units:
            pass
        else:
            print(i)
            
    new_cube = Cubelist.concatenate_cube()

    # Including time attributes
    iris.coord_categorisation.add_year(new_cube, 'time', name='year')
    iris.coord_categorisation.add_month(new_cube, 'time', name ='month')

    # Annual average
    aa_cube = new_cube.aggregated_by('year', iris.analysis.MEAN)

    return aa_cube

def global_total_percentage(cube, landfrac=None, latlon_cons=None, tropical=False):
    """
    Calculate global or tropical totals, with land fractions.
    """

    if landfrac is not None:
        try:
            cube.data = cube.data * (landfrac.data/100)
        except:
            landfrac = landfrac.extract(latlon_cons)
            cube.data = cube.data * (landfrac.data/100)

    if cube.coord('latitude').bounds is not None:
        pass
    else:
        cube.coord('latitude').guess_bounds()
        cube.coord('longitude').guess_bounds()

    if tropical==True:
        cube = cube.extract(iris.Constraint(latitude=lambda cell: -30 <= cell.point <= 30))
    else:
        pass

    weights = iris.analysis.cartography.area_weights(cube)
    cube_areaweight = cube.collapsed(['latitude', 'longitude'], iris.analysis.SUM, weights=weights)/1e12

    return cube_areaweight

def area_average(cube, region):
    """
    Area average of a cube over a specified region.
    """

    lon1, lon2, lat1, lat2 = region[0], region[1], region[2], region[3] 
    cube = cube.intersection(longitude=(lon1, lon2),latitude=(lat1, lat2))

    weights = iris.analysis.cartography.area_weights(cube)
    cube = cube.collapsed(['latitude','longitude'], iris.analysis.MEAN, weights=weights)

    return cube


In [ ]:
### BGC-COU approach with T* = 0, carbon stocks (Arora et al., 2020)
"""
beta = dC_bgc/Co2, where dC_bgc = cLand_bgc[TARGET_YEAR - 1] - cLand_piControl[equivalent_year]
gamma = (dC_cou - dC_bgc)/dT_cou, where dC_cou = cLand_cou[TARGET_YEAR - 1] - cLand_piControl[equivalent_year], and dT_cou = tas_cou[TARGET_YEAR - 1] - tas_piControl[equivalent_year]

Note: equivalent year in piControl isn't necessarily the same as TARGET_YEAR, as depends when CMIP6 model drifted off the parent simulation.
"""

# CMIP6 models
cmip6_models = [
    'ACCESS-ESM1-5',
    'BCC-CSM2-MR',
    'CanESM5',
    'CESM2',
    'CMCC-ESM2',
    'CNRM-ESM2-1',
    'EC-Earth3-CC',
    'GFDL-ESM4',
    'IPSL-CM6A-LR',
    'MIROC-ES2L',
    'MPI-ESM1-2-LR',
    'NorESM2-LM',
    'UKESM1-0-LL'
]
n_models = len(cmip6_models)

TARGET_YEAR = 70
INDEX_YEAR = TARGET_YEAR - 1
PREINDUSTRIAL_CO2 = 284.3
CO2_TARGET = PREINDUSTRIAL_CO2 * (1.01 ** INDEX_YEAR) - PREINDUSTRIAL_CO2

# Hard coded equivalent years for piControl for CMIP6
EQUIV_YEAR = {
    'ACCESS-ESM1-5':171, # 101 + 70
    'BCC-CSM2-MR':70,
    'CanESM5':70,
    'CESM2':570, #500 + 70
    'CMCC-ESM2':70,
    'CNRM-ESM2-1':70,
    'EC-Earth3-CC':70,
    'GFDL-ESM4':170, # 100 + 70
    'IPSL-CM6A-LR':90, # 20 + 70
    'MIROC-ES2L':70,
    'MPI-ESM1-2-LR':70,
    'NorESM2-LM':70, # year 1600
    'UKESM1-0-LL':180 # 110 + 70
}

results = {}

for model_i in range(n_models):
        model = cmip6_models[model_i]
        print(model)

        # Specific model PI_INDEX
        PI_INDEX = EQUIV_YEAR[model] - 1

        # load land fraction
        if model == 'EC-Earth3-CC':
                model_update = 'EC-Earth3'
                landfraction = iris.load_cube('/Volumes/Extreme SSD/DATA/cmip6_data/sftlf_fx_'+model_update+'_historical*')
        else:
                landfraction = iris.load_cube('/Volumes/Extreme SSD/DATA/cmip6_data/sftlf_fx_'+model+'_historical*')


        # bgc cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cVeg_Lmon_'+model+'_1pctCO2-bgc*', model)
                cVeg_bgc_cube = global_total_percentage(cVeg_bgc_cube, landfrac=landfraction, latlon_cons=None)
                # cSoil
                cSoil_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cSoil_Emon_'+model+'_1pctCO2-bgc*', model)
                cSoil_bgc_cube = global_total_percentage(cSoil_bgc_cube, landfrac=landfraction, latlon_cons=None)
                # cLitter
                cLitter_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cLitter_Lmon_'+model+'_1pctCO2-bgc*', model)
                cLitter_bgc_cube = global_total_percentage(cLitter_bgc_cube, landfrac=landfraction, latlon_cons=None)
                # 
                cLand_bgc_data = cVeg_bgc_cube.data + cSoil_bgc_cube.data + cLitter_bgc_cube.data
        else:
                # cLand
                cLand_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cLand_Emon_'+model+'_1pctCO2-bgc*', model)
                cLand_bgc_cube = global_total_percentage(cLand_bgc_cube, landfrac=landfraction, latlon_cons=None)
                cLand_bgc_data = cLand_bgc_cube.data

        # cou cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cVeg_Lmon_'+model+'_1pctCO2*', model)
                cVeg_cou_cube = global_total_percentage(cVeg_cou_cube, landfrac=landfraction, latlon_cons=None)
                # cSoil
                cSoil_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cSoil_Emon_'+model+'_1pctCO2*', model)
                cSoil_cou_cube = global_total_percentage(cSoil_cou_cube, landfrac=landfraction, latlon_cons=None)
                # cLitter
                cLitter_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cLitter_Lmon_'+model+'_1pctCO2*', model)
                cLitter_cou_cube = global_total_percentage(cLitter_cou_cube, landfrac=landfraction, latlon_cons=None)
                # 
                cLand_cou_data = cVeg_cou_cube.data + cSoil_cou_cube.data + cLitter_cou_cube.data
        else:
                # cLand
                cLand_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cLand_Emon_'+model+'_1pctCO2*', model)
                cLand_cou_cube = global_total_percentage(cLand_cou_cube, landfrac=landfraction, latlon_cons=None)
                cLand_cou_data = cLand_cou_cube.data

        # piControl cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cVeg_Lmon_'+model+'_piControl*', model)
                cVeg_piControl_cube = global_total_percentage(cVeg_piControl_cube, landfrac=landfraction, latlon_cons=None)
                # cSoil
                cSoil_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cSoil_Emon_'+model+'_piControl*', model)
                cSoil_piControl_cube = global_total_percentage(cSoil_piControl_cube, landfrac=landfraction, latlon_cons=None)
                # cLitter
                cLitter_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cLitter_Lmon_'+model+'_piControl*', model)
                cLitter_piControl_cube = global_total_percentage(cLitter_piControl_cube, landfrac=landfraction, latlon_cons=None)
                # 
                cLand_piControl_data = cVeg_piControl_cube.data + cSoil_piControl_cube.data + cLitter_piControl_cube.data
        elif model == 'EC-Earth3-CC' or model == 'GFDL-ESM4':
                print(f"Skipping {model}")
                pass
        else:
                # cLand
                cLand_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cLand_Emon_'+model+'_piControl*', model)
                cLand_piControl_cube = global_total_percentage(cLand_piControl_cube, landfrac=landfraction, latlon_cons=None)
                cLand_piControl_data = cLand_piControl_cube.data

        # cou tas
        tas_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/tas_Amon_'+model+'_1pctCO2*', model)
        tas_cou_cube = area_average(tas_cou_cube - 273.15, region=[0, 360, -90,  90])
        tas_cou_data = tas_cou_cube.data
        # 21 year mean centered around TARGET_YEAR
        tas_cou_slice = tas_cou_data[INDEX_YEAR-10:INDEX_YEAR+11]
        tas_cou  = np.nanmean(tas_cou_slice)

        # piControl tas
        tas_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/tas_Amon_'+model+'_piControl*', model)
        tas_piControl_cube = area_average(tas_piControl_cube - 273.15, region=[0, 360, -90,  90])
        tas_piControl_data = tas_piControl_cube.data
        # 21 year mean centered around PI_INDEX
        tas_piControl_slice = tas_piControl_data[PI_INDEX-10:PI_INDEX+11]
        tas_piControl  = np.nanmean(tas_piControl_slice)

        if model == 'EC-Earth3-CC' or model == 'GFDL-ESM4':
                # beta
                dC_bgc = cLand_bgc_data[INDEX_YEAR] - cLand_bgc_data[0]
                beta = dC_bgc/CO2_TARGET

                # gamma
                dC_cou = cLand_cou_data[INDEX_YEAR] - cLand_cou_data[0]
                dT_cou = tas_cou - tas_piControl
                dC_res = dC_cou - dC_bgc
                gamma = dC_res/dT_cou
        else:
                # beta
                dC_bgc = cLand_bgc_data[INDEX_YEAR] - cLand_piControl_data[PI_INDEX]
                beta = dC_bgc/CO2_TARGET

                # gamma
                dC_cou = cLand_cou_data[INDEX_YEAR] - cLand_piControl_data[PI_INDEX]
                dT_cou = tas_cou - tas_piControl
                dC_res = dC_cou - dC_bgc
                gamma = dC_res/dT_cou

        results[model] = {
            "beta": beta,
            "gamma": gamma,
        }

df = pd.DataFrame(results).T
df.to_csv("CMIP6_beta_gamma_global.csv")
print(df)


ACCESS-ESM1-5


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/fileformats/cf.py:939: IrisCfMissingVarWarning: Missing CF-netCDF measure variable 'areacella', referenced by netCDF variable 'sftlf'
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precisi

HERE


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

BCC-CSM2-MR


...

UKESM1-0-LL


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

HERE


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

                   beta      gamma
ACCESS-ESM1-5  0.722859 -10.332458
BCC-CSM2-MR    2.190957 -91.400160
CanESM5        1.436439  -6.301729
CESM2          0.985735 -13.730428
CMCC-ESM2      0.395613  -9.394278
CNRM-ESM2-1    1.380901 -46.169163
EC-Earth3-CC   0.867681 -13.890725
GFDL-ESM4      1.068781 -44.401649
IPSL-CM6A-LR   0.963136  -7.404462
MIROC-ES2L     1.420855 -57.124401
MPI-ESM1-2-LR  1.077715  -1.203908
NorESM2-LM     0.945042 -16.493495
UKESM1-0-LL    1.054999 -22.955561


In [ ]:
### TROPICS

# CMIP6 models
cmip6_models = [
    'ACCESS-ESM1-5',
    'BCC-CSM2-MR',
    'CanESM5',
    'CESM2',
    'CMCC-ESM2',
    'CNRM-ESM2-1',
    'EC-Earth3-CC',
    'GFDL-ESM4',
    'IPSL-CM6A-LR',
    'MIROC-ES2L',
    'MPI-ESM1-2-LR',
    'NorESM2-LM',
    'UKESM1-0-LL'
]
n_models = len(cmip6_models)

TARGET_YEAR = 70
INDEX_YEAR = TARGET_YEAR - 1
PREINDUSTRIAL_CO2 = 284.3
CO2_TARGET = PREINDUSTRIAL_CO2 * (1.01 ** INDEX_YEAR) - PREINDUSTRIAL_CO2

# Hard coded equivalent years for piControl for CMIP6
EQUIV_YEAR = {
    'ACCESS-ESM1-5':70,
    'BCC-CSM2-MR':70,
    'CanESM5':70, # 3351 + 70
    'CESM2':570, #500 + 70
    'CMCC-ESM2':70,
    'CNRM-ESM2-1':70,
    'EC-Earth3-CC':70,
    'GFDL-ESM4':170, # 100 + 70
    'IPSL-CM6A-LR':90, # 20 + 70
    'MIROC-ES2L':70,
    'MPI-ESM1-2-LR':70,
    'NorESM2-LM':70, # 1600 + 70
    'UKESM1-0-LL':180 # 110 + 70
}

results = {}

for model_i in range(n_models):
        model = cmip6_models[model_i]
        print(model)

        # Specific model PI_INDEX
        PI_INDEX = EQUIV_YEAR[model] - 1

        # load land fraction
        if model == 'EC-Earth3-CC':
                model_update = 'EC-Earth3'
                landfraction = iris.load_cube('/Volumes/Extreme SSD/DATA/cmip6_data/sftlf_fx_'+model_update+'_historical*')
        else:
                landfraction = iris.load_cube('/Volumes/Extreme SSD/DATA/cmip6_data/sftlf_fx_'+model+'_historical*')


        # bgc cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cVeg_Lmon_'+model+'_1pctCO2-bgc*', model)
                cVeg_bgc_cube = global_total_percentage(cVeg_bgc_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cSoil
                cSoil_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cSoil_Emon_'+model+'_1pctCO2-bgc*', model)
                cSoil_bgc_cube = global_total_percentage(cSoil_bgc_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cLitter
                cLitter_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cLitter_Lmon_'+model+'_1pctCO2-bgc*', model)
                cLitter_bgc_cube = global_total_percentage(cLitter_bgc_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # 
                cLand_bgc_data = cVeg_bgc_cube.data + cSoil_bgc_cube.data + cLitter_bgc_cube.data
        else:
                # cLand
                cLand_bgc_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2-bgc/cLand_Emon_'+model+'_1pctCO2-bgc*', model)
                cLand_bgc_cube = global_total_percentage(cLand_bgc_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                cLand_bgc_data = cLand_bgc_cube.data

        # cou cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cVeg_Lmon_'+model+'_1pctCO2*', model)
                cVeg_cou_cube = global_total_percentage(cVeg_cou_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cSoil
                cSoil_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cSoil_Emon_'+model+'_1pctCO2*', model)
                cSoil_cou_cube = global_total_percentage(cSoil_cou_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cLitter
                cLitter_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cLitter_Lmon_'+model+'_1pctCO2*', model)
                cLitter_cou_cube = global_total_percentage(cLitter_cou_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # 
                cLand_cou_data = cVeg_cou_cube.data + cSoil_cou_cube.data + cLitter_cou_cube.data
        else:
                # cLand
                cLand_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/cLand_Emon_'+model+'_1pctCO2*', model)
                cLand_cou_cube = global_total_percentage(cLand_cou_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                cLand_cou_data = cLand_cou_cube.data

        # piControl cLand
        if model == 'BCC-CSM2-MR':
                # cVeg
                cVeg_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cVeg_Lmon_'+model+'_piControl*', model)
                cVeg_piControl_cube = global_total_percentage(cVeg_piControl_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cSoil
                cSoil_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cSoil_Emon_'+model+'_piControl*', model)
                cSoil_piControl_cube = global_total_percentage(cSoil_piControl_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # cLitter
                cLitter_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cLitter_Lmon_'+model+'_piControl*', model)
                cLitter_piControl_cube = global_total_percentage(cLitter_piControl_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                # 
                cLand_piControl_data = cVeg_piControl_cube.data + cSoil_piControl_cube.data + cLitter_piControl_cube.data
        elif model == 'EC-Earth3-CC' or model == 'GFDL-ESM4':
                print(f"Skipping {model}")
                pass
        else:
                # cLand
                cLand_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/cLand_Emon_'+model+'_piControl*', model)
                cLand_piControl_cube = global_total_percentage(cLand_piControl_cube, landfrac=landfraction, latlon_cons=None, tropical=True)
                cLand_piControl_data = cLand_piControl_cube.data

        # cou tas
        tas_cou_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_1pctCO2/tas_Amon_'+model+'_1pctCO2*', model)
        tas_cou_cube = area_average(tas_cou_cube - 273.15, region=[0, 360, -90,  90])
        tas_cou_data = tas_cou_cube.data
        # 21 year mean centered around TARGET_YEAR
        tas_cou_slice = tas_cou_data[INDEX_YEAR-10:INDEX_YEAR+11]
        tas_cou  = np.nanmean(tas_cou_slice)

        # piControl tas
        tas_piControl_cube = cmip6_tcre_processing('/Volumes/Extreme SSD/DATA/CMIP_piControl/tas_Amon_'+model+'_piControl*', model)
        tas_piControl_cube = area_average(tas_piControl_cube - 273.15, region=[0, 360, -90,  90])
        tas_piControl_data = tas_piControl_cube.data
        # 21 year mean centered around PI_INDEX
        tas_piControl_slice = tas_piControl_data[PI_INDEX-10:PI_INDEX+11]
        tas_piControl  = np.nanmean(tas_piControl_slice)

        if model == 'EC-Earth3-CC' or model == 'GFDL-ESM4':
                # beta
                dC_bgc = cLand_bgc_data[INDEX_YEAR] - cLand_bgc_data[0]
                beta = dC_bgc/CO2_TARGET

                # gamma
                dC_cou = cLand_cou_data[INDEX_YEAR] - cLand_cou_data[0]
                dT_cou = tas_cou - tas_piControl
                dC_res = dC_cou - dC_bgc
                gamma = dC_res/dT_cou
        else:
                # beta
                dC_bgc = cLand_bgc_data[INDEX_YEAR] - cLand_piControl_data[PI_INDEX]
                beta = dC_bgc/CO2_TARGET

                # gamma
                dC_cou = cLand_cou_data[INDEX_YEAR] - cLand_piControl_data[PI_INDEX]
                dT_cou = tas_cou - tas_piControl
                dC_res = dC_cou - dC_bgc
                gamma = dC_res/dT_cou

        results[model] = {
            "beta": beta,
            "gamma": gamma,
        }

df = pd.DataFrame(results).T
df.to_csv("CMIP6_beta_gamma_tropics.csv")
print(df)

ACCESS-ESM1-5


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/fileformats/cf.py:939: IrisCfMissingVarWarning: Missing CF-netCDF measure variable 'areacella', referenced by netCDF variable 'sftlf'
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precisi

HERE


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

BCC-CSM2-MR


...

UKESM1-0-LL


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

HERE


/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microseconds = True`.
  warnings.warn(message, category=FutureWarning)
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/analysis/cartography.py:441: IrisDefaultingWarning: Using DEFAULT_SPHERICAL_EARTH_RADIUS.
  warnings.warn(
/Users/rebeccamayvarney/miniforge3/envs/rmv/lib/python3.13/site-packages/iris/common/mixin.py:212: FutureWarning: You are using legacy date precision for Iris units - max precision is seconds. In future, Iris will use microsecond precision - available since cf-units version 3.3 - which may affect core behaviour. To opt-in to the new behaviour, set `iris.FUTURE.date_microsec

                   beta      gamma
ACCESS-ESM1-5  0.378504 -14.368197
BCC-CSM2-MR    1.097014 -42.340114
CanESM5        1.196410 -25.134087
CESM2          0.709356 -14.586917
CMCC-ESM2      0.260276  -2.701159
CNRM-ESM2-1    0.557704 -32.818542
EC-Earth3-CC   0.598599 -12.194137
GFDL-ESM4      0.649849 -37.336638
IPSL-CM6A-LR   0.597862 -11.730475
MIROC-ES2L     0.883228 -47.628697
MPI-ESM1-2-LR  0.805636   1.408490
NorESM2-LM     0.686995 -17.465520
UKESM1-0-LL    0.685961 -25.413400
